# PDD Mobility Scanner — Video + Sensor Sync & Object Detection

Upload `trail_data.csv` and `trail_video.mp4` from the SD card.
This notebook:
1. Syncs video frames to CSV sensor data using `rec_start_ms`
2. Runs YOLO object detection on each frame
3. Pairs detected objects with GPS coordinates and timestamps
4. Exports a combined dataset and annotated frames

## Step 1: Install dependencies

In [ ]:
!pip install -q ultralytics
import pandas as pd
import numpy as np
import cv2
from ultralytics import YOLO
from google.colab import files
from IPython.display import display, Image as IPImage
import matplotlib.pyplot as plt
import os

## Step 2: Upload files

In [ ]:
print("Upload trail_data.csv and trail_video.mp4")
uploaded = files.upload()

csv_file = [f for f in uploaded.keys() if f.endswith('.csv')][0]
video_file = [f for f in uploaded.keys() if f.endswith('.mp4')][0]
print(f"CSV: {csv_file}")
print(f"Video: {video_file}")

## Step 3: Load CSV and extract rec_start_ms

In [ ]:
# Read rec_start_ms from the comment line at the top of the CSV
with open(csv_file) as f:
    first_line = f.readline().strip()
rec_start_ms = int(first_line.split('=')[1])
print(f"Video recording started at ms={rec_start_ms}")

# Load sensor data (skip the comment line)
df = pd.read_csv(csv_file, comment='#')

# Compute video time for each sensor row
df['video_sec'] = (df['ms'] - rec_start_ms) / 1000.0

print(f"Loaded {len(df)} sensor rows")
print(f"Recording duration: {df['video_sec'].max():.1f}s")
print(f"GPS fixes: {df['lat'].notna().sum()} rows")
df[['ms', 'video_sec', 'utc', 'lat', 'lng']].head(10)

## Step 4: Extract video frames and sync to sensor data

For each video frame, find the closest sensor row by timestamp to get GPS + IMU data.

In [ ]:
cap = cv2.VideoCapture(video_file)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {total_frames} frames at {fps} FPS ({total_frames/fps:.1f}s)")

# Extract all frames
os.makedirs('frames', exist_ok=True)
frames = []
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    video_sec = frame_idx / fps
    ms_at_frame = rec_start_ms + int(video_sec * 1000)

    # Find closest sensor row
    closest_idx = (df['ms'] - ms_at_frame).abs().idxmin()
    row = df.loc[closest_idx]

    frames.append({
        'frame': frame_idx,
        'video_sec': video_sec,
        'ms': ms_at_frame,
        'sensor_ms': row['ms'],
        'sync_offset_ms': abs(row['ms'] - ms_at_frame),
        'utc': row.get('utc', ''),
        'lat': row.get('lat', np.nan),
        'lng': row.get('lng', np.nan),
        'alt': row.get('alt', np.nan),
        'speed': row.get('speed', np.nan),
    })

    # Save frame image
    cv2.imwrite(f'frames/frame_{frame_idx:04d}.jpg', frame)
    frame_idx += 1

cap.release()

frames_df = pd.DataFrame(frames)
print(f"Extracted {len(frames_df)} frames")
print(f"Average sync offset: {frames_df['sync_offset_ms'].mean():.1f}ms")
frames_df.head(10)

## Step 5: Run YOLO object detection on each frame

In [ ]:
# Load YOLO model (downloads automatically on first run)
model = YOLO('yolov8n.pt')

# Run detection on all frames
detections = []

for i, row in frames_df.iterrows():
    img_path = f'frames/frame_{row["frame"]:04d}.jpg'
    results = model(img_path, verbose=False)

    frame_objects = []
    for r in results:
        for box in r.boxes:
            cls_name = model.names[int(box.cls)]
            conf = float(box.conf)
            frame_objects.append({
                'frame': row['frame'],
                'video_sec': row['video_sec'],
                'utc': row['utc'],
                'lat': row['lat'],
                'lng': row['lng'],
                'object': cls_name,
                'confidence': conf,
                'bbox': box.xyxy[0].tolist(),
            })

    if not frame_objects:
        # Log frames with no detections too
        detections.append({
            'frame': row['frame'],
            'video_sec': row['video_sec'],
            'utc': row['utc'],
            'lat': row['lat'],
            'lng': row['lng'],
            'object': None,
            'confidence': None,
            'bbox': None,
        })
    else:
        detections.extend(frame_objects)

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(frames_df)} frames...")

det_df = pd.DataFrame(detections)
obj_counts = det_df['object'].value_counts()
print(f"\nTotal detections: {det_df['object'].notna().sum()}")
print(f"Unique object types: {det_df['object'].nunique()}")
print(f"\nObject counts:")
print(obj_counts)

## Step 6: Show annotated frames

Display a sample of frames with detections overlaid and GPS coordinates.

In [ ]:
# Pick frames that have detections
frames_with_objects = det_df[det_df['object'].notna()]['frame'].unique()
sample_frames = frames_with_objects[:8]  # show up to 8

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, frame_num in enumerate(sample_frames):
    if i >= len(axes):
        break

    img_path = f'frames/frame_{frame_num:04d}.jpg'
    results = model(img_path, verbose=False)
    annotated = results[0].plot()

    # Get GPS for this frame
    frame_info = frames_df[frames_df['frame'] == frame_num].iloc[0]
    lat = frame_info['lat']
    lng = frame_info['lng']
    sec = frame_info['video_sec']

    axes[i].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    title = f"Frame {frame_num} ({sec:.0f}s)"
    if pd.notna(lat):
        title += f"\n{lat:.5f}, {lng:.5f}"
    axes[i].set_title(title, fontsize=9)
    axes[i].axis('off')

# Hide unused subplots
for j in range(len(sample_frames), len(axes)):
    axes[j].axis('off')

plt.suptitle('Detected Objects with GPS Coordinates', fontsize=14)
plt.tight_layout()
plt.show()

## Step 7: Detection timeline

In [ ]:
# Objects detected over time
obj_timeline = det_df[det_df['object'].notna()].copy()

if len(obj_timeline) > 0:
    top_objects = obj_timeline['object'].value_counts().head(5).index
    
    fig, ax = plt.subplots(figsize=(14, 4))
    for obj_name in top_objects:
        obj_rows = obj_timeline[obj_timeline['object'] == obj_name]
        ax.scatter(obj_rows['video_sec'], [obj_name] * len(obj_rows),
                   alpha=0.6, s=30, label=obj_name)
    ax.set_xlabel('Video time (s)')
    ax.set_title('Object detections over time')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No objects detected.")

## Step 8: Export results

In [ ]:
# Save full detection log
export = det_df.drop(columns=['bbox']).copy()
export.to_csv('detections.csv', index=False)
print(f"Saved {len(export)} rows to detections.csv")

# Summary: unique objects per GPS location
gps_objects = obj_timeline.dropna(subset=['lat', 'lng']).groupby(
    ['frame', 'video_sec', 'lat', 'lng']
)['object'].apply(list).reset_index()
gps_objects.columns = ['frame', 'video_sec', 'lat', 'lng', 'objects']
gps_objects.to_csv('gps_detections.csv', index=False)
print(f"Saved {len(gps_objects)} geolocated frames to gps_detections.csv")

files.download('detections.csv')
files.download('gps_detections.csv')

gps_objects.head(10)